# Phase 1 — first pizza chart (exploratory)

Goal of this notebook: pull one completed season, inspect the raw player
table, and (later) draw **one** static pizza chart.

This is **not** the pipeline. No `src/soccer_pizza_charts/` modules, no JSON export, no UI.

**FBref (kept, not deleted):** Big 5 Combined `2526` is cached under
`data/data/FBref/`. FBref dropped Opta's advanced-stats feed in Jan 2026, so
those tables have no xG / progression. We reuse them in Phase 2 for the
counting stats they still have (goals, shots, tackles won, …).

**Current fetch:** Understat via `soccerdata`, Premier League 2025/26, for
xG-side player season stats.

**Checkpoint:** still no per-90 rates, percentiles, or chart.


## 1. Project paths

Resolve the repo root from the notebook location so this works whether the
kernel's working directory is the repo root or `notebooks/`. Create `data/`
and `outputs/` if they are missing. (`data/` is gitignored; it is only a
local cache.)


In [1]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "pyproject.toml").exists():
    PROJECT_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "pyproject.toml").exists():
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    raise FileNotFoundError(
        f"Could not find pyproject.toml from cwd={NOTEBOOK_DIR}"
    )

DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:    ", DATA_DIR)
print("OUTPUTS_DIR: ", OUTPUTS_DIR)


PROJECT_ROOT: /Users/sargo/soccer-pizza-charts
DATA_DIR:     /Users/sargo/soccer-pizza-charts/data
OUTPUTS_DIR:  /Users/sargo/soccer-pizza-charts/outputs


## 2. Cache into the project's `data/` directory

`soccerdata` reads `SOCCERDATA_DIR` **at import time**. The default is
`~/soccerdata`. We set the env var to this project's `data/` **before**
importing the library so HTML caches and logs stay with the repo (and stay
gitignored).

Because soccerdata stores files under `{SOCCERDATA_DIR}/data/...`, the
FBref HTML cache will land at `data/data/FBref/` and logs at `data/logs/`.
That nested `data/data` is the library's layout, not a mistake.


In [2]:
import os

# Must happen before `import soccerdata`.
os.environ["SOCCERDATA_DIR"] = str(DATA_DIR)

import soccerdata as sd
from soccerdata._config import BASE_DIR, DATA_DIR as SOCCERDATA_DATA_DIR, LOGS_DIR

print("SOCCERDATA_DIR env:", os.environ["SOCCERDATA_DIR"])
print("soccerdata BASE_DIR:", BASE_DIR)
print("soccerdata DATA_DIR:", SOCCERDATA_DATA_DIR)
print("soccerdata LOGS_DIR:", LOGS_DIR)

def _is_inside(path: Path, parent: Path) -> bool:
    try:
        path.resolve().relative_to(parent.resolve())
        return True
    except ValueError:
        return False

print()
print("BASE_DIR is inside project data/:", _is_inside(BASE_DIR, DATA_DIR))
print("DATA_DIR is inside project data/:", _is_inside(SOCCERDATA_DATA_DIR, DATA_DIR))
print("LOGS_DIR is inside project data/:", _is_inside(LOGS_DIR, DATA_DIR))
if not _is_inside(BASE_DIR, DATA_DIR):
    raise RuntimeError(
        f"Cache escaped the project: BASE_DIR={BASE_DIR} is not under {DATA_DIR}"
    )


[08/22/26 15:56:44] INFO     No custom team name replacements found. You can configure these in       ]8;id=10269960;file:///Users/sargo/soccer-pizza-charts/.venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=10269961;file:///Users/sargo/soccer-pizza-charts/.venv/lib/python3.12/site-packages/soccerdata/_config.py#91\91]8;;\
                             /Users/sargo/soccer-pizza-charts/data/config/teamname_replacements.json.              

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=10269969;file:///Users/sargo/soccer-pizza-charts/.venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=10269970;file:///Users/sargo/soccer-pizza-charts/.venv/lib/python3.12/site-packages/soccerdata/_config.py#189\189]8;;\
                             /Users/sargo/soccer-pizza-charts/data/config/league_dict.json.                        

SOCCERDATA_DIR env:

/Users/sargo/soccer-pizza-charts/data

soccerdata BASE_DIR:

...

True

DATA_DIR is inside project data/:

True

LOGS_DIR is inside project data/:

True

## 3. FBref scraper — Big 5 Combined, visible Chrome

**Big 5 European Leagues Combined**, season **2025–26**. soccerdata accepts
several season-string formats (`'16-17'`, `2016`, `'2016-17'`); we use
`'2526'`.

The Combined player `standard` table is a different FBref page than a
single-league scrape (soccerdata reads `table#stats_standard` instead of
the comment-hidden league table). That is the table that usually carries
Expected (xG / npxG / xAG) and Progression (PrgC / PrgP / PrgR).

soccerdata **does** expose a headless toggle: `FBref(..., headless=False)`.
That is also FBref's constructor default (the class docstring still says
`default: True`; the actual `__init__` default is `False`).

**How the browser launches** (library behavior — we are not changing it):

- Constructing `FBref` immediately starts Chrome via SeleniumBase:
  `sb.Driver(uc=True, headless=self.headless, ...)`.
- The first *uncached* request opens FBref. If Cloudflare's "verify you
  are human" page appears, complete the checkbox in that window. soccerdata
  detects the challenge, waits, and retries a few times (about 10s each).
- HTML is then written under `fbref.data_dir` (`data/data/FBref/`). Later
  calls reuse those files and should not re-hit the network.

We are not adding stealth drivers, proxies, or extra captcha tooling. If a
clickable Chrome window never appears, stop — the fallback is a manual CSV
export from FBref.


In [3]:
LEAGUE = "Big 5 European Leagues Combined"
SEASON = "2526"

# Visible Chrome: FBref's own default is already False; set it explicitly.
fbref = sd.FBref(leagues=LEAGUE, seasons=SEASON, headless=False)

print("leagues:", fbref.leagues)
print("seasons:", list(fbref.seasons))
print("fbref.headless:", fbref.headless)
print("fbref.data_dir:", fbref.data_dir)
print("fbref.data_dir inside project data/:", _is_inside(fbref.data_dir, DATA_DIR))
if fbref.headless:
    raise RuntimeError(
        "Chrome is headless; Cloudflare cannot be completed by hand. Stop."
    )
fbref.data_dir


                    INFO     Saving cached data to /Users/sargo/soccer-pizza-charts/data/data/FBref  ]8;id=10269979;file:///Users/sargo/soccer-pizza-charts/.venv/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=10269980;file:///Users/sargo/soccer-pizza-charts/.venv/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\

leagues:

['Big 5 European Leagues Combined']

seasons:

['2526']

...

fbref.data_dir:

/Users/sargo/soccer-pizza-charts/data/data/FBref

fbref.data_dir inside project data/:

True

PosixPath('/Users/sargo/soccer-pizza-charts/data/data/FBref')

## 4a. What `stat_type` values does this install actually accept?

Do not assume. The docstring / signature of `read_player_season_stats` is
the source of truth for soccerdata 1.9.1.


In [4]:
import inspect

print("signature:", inspect.signature(sd.FBref.read_player_season_stats))
print()
print(inspect.getdoc(sd.FBref.read_player_season_stats))


signature:

(self, stat_type: str = 'standard') -> pandas.DataFrame

Retrieve players from the datasource for the selected leagues and seasons.

The following stat types are available:
    * 'standard'
    * 'shooting'
    * 'playing_time'
    * 'keeper'
    * 'misc'

Parameters
----------
stat_type :str
    Type of stats to retrieve.

Raises
------
TypeError
    If ``stat_type`` is not valid.

Returns
-------
pd.DataFrame

## 4b. Restricted list (already confirmed)

soccerdata 1.9.1 rejects `passing`, `defense`, and `possession` at the
allowlist check (`TypeError`, before any download). We will not request
them and will not monkeypatch.

The next cells fetch only the three outfield tables the library actually
serves: `standard`, `shooting`, `misc`.


In [5]:
print("Will fetch:    ", ["standard", "shooting", "misc"])
print("Will not fetch:", ["passing", "defense", "possession"])


Will fetch:    

['standard', 'shooting', 'misc']

Will not fetch:

['passing', 'defense', 'possession']

## 4c. Fetch the three supported player tables

Native fetches only: `standard`, `shooting`, `misc`. No passing / defense /
possession, and no allowlist patch.

The first uncached call hits the network. `leagues.html` may already be
cached from the Premier League probe; Big 5 season + player pages are new.
Complete Cloudflare in the visible Chrome window if it appears. After that,
files under `fbref.data_dir` mean a retry should not re-download.

**Do not merge yet.** The next cell inspects the `standard` table alone for
Expected and Progression columns — that is the reason we switched to Big 5.


In [6]:
from collections import OrderedDict
import traceback

import pandas as pd

REQUESTED_STAT_TYPES = ["standard", "shooting", "misc"]

fetched: dict[str, pd.DataFrame] = OrderedDict()
fetch_errors: dict[str, str] = OrderedDict()

for stat_type in REQUESTED_STAT_TYPES:
    print(f"--- fetching {stat_type!r} ---")
    try:
        frame = fbref.read_player_season_stats(stat_type=stat_type)
        fetched[stat_type] = frame
        print(f"  ok  shape={frame.shape}  column nlevels={frame.columns.nlevels}")
        print(f"  index names: {list(frame.index.names)}")
        print(f"  columns: {frame.columns.tolist()}")
    except Exception as exc:
        fetch_errors[stat_type] = f"{type(exc).__name__}: {exc}"
        print(f"  FAIL  {fetch_errors[stat_type]}")
        traceback.print_exc()
    print()

print("succeeded:", list(fetched))
print("failed:   ", dict(fetch_errors))


--- fetching 'standard' ---

  ok  shape=(2839, 21)  column nlevels=2

  index names: ['league', 'season', 'team', 'player']

  columns: [('nation', ''), ('pos', ''), ('age', ''), ('born', ''), ('Playing Time', 'MP'), ('Playing Time', 'Starts'), ('Playing Time', 'Min'), ('Playing Time', '90s'), ('Performance', 'Gls'), ('Performance', 'Ast'), ('Performance', 'G+A'), ('Performance', 'G-PK'), ('Performance', 'PK'), ('Performance', 'PKatt'), ('Performance', 'CrdY'), ('Performance', 'CrdR'), ('Per 90 Minutes', 'Gls'), ('Per 90 Minutes', 'Ast'), ('Per 90 Minutes', 'G+A'), ('Per 90 Minutes', 'G-PK'), ('Per 90 Minutes', 'G+A-PK')]

--- fetching 'shooting' ---

...

  columns: [('nation', ''), ('pos', ''), ('age', ''), ('born', ''), ('90s', ''), ('Performance', 'CrdY'), ('Performance', 'CrdR'), ('Performance', '2CrdY'), ('Performance', 'Fls'), ('Performance', 'Fld'), ('Performance', 'Off'), ('Performance', 'Crs'), ('Performance', 'Int'), ('Performance', 'TklW'), ('Performance', 'PKwon'), ('Performance', 'PKcon'), ('Performance', 'OG')]

succeeded:

['standard', 'shooting', 'misc']

failed:   

{}

## 4c-check. Does Big 5 `standard` include Expected and Progression?

This is the reason for the switch. Print the **full** `standard` column list
before any flatten or merge, and say explicitly whether Expected
(`xG`, `npxG`, `xAG`) and Progression (`PrgC`, `PrgP`, `PrgR`) are present.

If they are still missing, stop — that is a parsing problem, not a reason
to keep merging.


In [7]:
HAS_XG_AND_PROGRESSION = False
standard = fetched.get("standard")

if standard is None:
    print("No standard table fetched; cannot check Expected / Progression.")
else:
    print("standard shape:", standard.shape)
    print("index names:", list(standard.index.names))
    print()
    print("FULL standard columns (before flatten/merge):")
    for i, col in enumerate(standard.columns, start=1):
        print(f"  {i:2d}. {col}")

    if isinstance(standard.columns, pd.MultiIndex):
        groups = [str(c[0]) for c in standard.columns]
        leaves = [str(c[-1]) for c in standard.columns]
    else:
        groups = [""] * len(standard.columns)
        leaves = [str(c) for c in standard.columns]

    expected_wanted = ["xG", "npxG", "xAG"]
    prog_wanted = ["PrgC", "PrgP", "PrgR"]
    expected_hits = [n for n in expected_wanted if n in leaves]
    prog_hits = [n for n in prog_wanted if n in leaves]
    expected_group = any("expected" in g.lower() for g in groups)
    prog_group = any("progress" in g.lower() for g in groups)

    print()
    print("Expected group present:", expected_group)
    print("Expected leaves found: ", expected_hits or "NONE")
    print("Expected leaves missing:", [n for n in expected_wanted if n not in leaves] or "none")
    print()
    print("Progression group present:", prog_group)
    print("Progression leaves found: ", prog_hits or "NONE")
    print("Progression leaves missing:", [n for n in prog_wanted if n not in leaves] or "none")

    HAS_XG_AND_PROGRESSION = len(expected_hits) == len(expected_wanted) and len(prog_hits) == len(prog_wanted)
    print()
    if HAS_XG_AND_PROGRESSION:
        print("RESULT: Big 5 standard table HAS Expected + Progression. Proceed to merge.")
    else:
        print("RESULT: STOP. Big 5 standard still lacks xG/progression.")
        print("Treat this as a soccerdata parsing issue, not a missing-page issue.")


standard shape:

(2839, 21)

index names:

['league', 'season', 'team', 'player']

FULL standard columns (before flatten/merge):

...

NONE

Progression leaves missing:

['PrgC', 'PrgP', 'PrgR']

RESULT: STOP. Big 5 standard still lacks xG/progression.

Treat this as a soccerdata parsing issue, not a missing-page issue.

## 4d. Merge standard + shooting + misc

Only run if the Big 5 `standard` table has Expected + Progression.

soccerdata's Big 5 player tables typically have:

- a **row MultiIndex**: `league` / `season` / `team` / `player`. `league` is
  the mapped competition (`ENG-Premier League`, `ESP-La Liga`, …), **not**
  the Combined label. Keep it — we filter on it (e.g. Salah at Liverpool).
- **column MultiIndex**: FBref group (e.g. `Expected`) / stat (e.g. `npxG`).

Flattening: drop empty / `Unnamed:*` levels, join the rest with `_`, then
snake_case. Join later tables onto `standard` on the row index. Columns
that already exist on the left are skipped so we do not duplicate `age` /
`pos` / playing-time fields.


In [8]:
def flatten_stat_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Collapse FBref (group, stat) MultiIndex columns to one snake_case name."""
    out = df.copy()
    if not isinstance(out.columns, pd.MultiIndex):
        out.columns = [_to_snake(str(c)) for c in out.columns]
        return out

    names: list[str] = []
    for col in out.columns:
        parts = []
        for level in col:
            text = str(level).strip()
            if not text or text.lower().startswith("unnamed"):
                continue
            parts.append(text)
        joined = " ".join(parts) if parts else " ".join(str(x) for x in col)
        names.append(_to_snake(joined))
    out.columns = names
    return out


def _to_snake(name: str) -> str:
    name = (
        name.strip()
        .replace("%", " pct")
        .replace("+", " plus ")
        .replace("/", " ")
        .replace("-", " ")
    )
    return "_".join(name.lower().split())


def flatten_stat_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Collapse FBref (group, stat) MultiIndex columns to one snake_case name."""
    out = df.copy()
    if not isinstance(out.columns, pd.MultiIndex):
        out.columns = [_to_snake(str(c)) for c in out.columns]
        return out

    names: list[str] = []
    for col in out.columns:
        parts = []
        for level in col:
            text = str(level).strip()
            if not text or text.lower().startswith("unnamed"):
                continue
            parts.append(text)
        joined = " ".join(parts) if parts else " ".join(str(x) for x in col)
        names.append(_to_snake(joined))
    out.columns = names
    return out


def _to_snake(name: str) -> str:
    name = (
        name.strip()
        .replace("%", " pct")
        .replace("+", " plus ")
        .replace("/", " ")
        .replace("-", " ")
    )
    return "_".join(name.lower().split())


flattened: dict[str, pd.DataFrame] = OrderedDict()
players = None

if not HAS_XG_AND_PROGRESSION:
    print("Skipping flatten/merge — Big 5 standard lacks Expected/Progression.")
else:
    for stat_type, frame in fetched.items():
        flat = flatten_stat_columns(frame)
        flat = flat.loc[:, ~flat.columns.duplicated()]
        flattened[stat_type] = flat
        print(f"{stat_type}: {flat.shape} -> {list(flat.columns)}")


Skipping flatten/merge — Big 5 standard lacks Expected/Progression.

In [9]:
players = None

if not HAS_XG_AND_PROGRESSION or not flattened:
    print("No merge — either the xG/progression check failed or no tables were fetched.")
    if fetch_errors:
        print("Fetch errors:")
        for stat_type, err in fetch_errors.items():
            print(f"  {stat_type}: {err}")
else:
    # Start from standard. Later tables contribute only columns that are not
    # already present, so identity fields and overlapping FBref stats stay unique.
    # Row index keeps league/season/team/player so we can filter to one competition.
    merged = next(iter(flattened.values())).copy()
    for stat_type, frame in list(flattened.items())[1:]:
        new_cols = [c for c in frame.columns if c not in merged.columns]
        dropped = [c for c in frame.columns if c in merged.columns]
        print(f"joining {stat_type}: adding {new_cols}")
        print(f"joining {stat_type}: already present, skipped {dropped}")
        merged = merged.join(frame[new_cols], how="outer")
    players = merged
    print("merged index names:", list(players.index.names))
    display(players)


No merge — either the xG/progression check failed or no tables were fetched.

## 4e. Inspect the merged player table

Shape, full column list, dtypes, player count, position distribution, and
Mohamed Salah at Liverpool / Premier League. Then re-check `docs/metrics.md`
against this Big 5 table.


In [10]:
if players is None:
    print("No merged table — fetch did not succeed, or xG/progression check failed.")
else:
    print("shape:", players.shape)
    print("n players:", len(players))
    print("index names:", list(players.index.names))
    print()
    print("league distribution:")
    print(players.index.get_level_values("league").value_counts().to_string())
    print()
    print("position distribution (pos):")
    print(players["pos"].value_counts(dropna=False).to_string())
    print()
    print("columns:")
    for i, col in enumerate(players.columns, start=1):
        print(f"  {i:2d}. {col}")
    print()
    print("dtypes:")
    print(players.dtypes.to_string())


No merged table — fetch did not succeed, or xG/progression check failed.

In [11]:
if players is None:
    print("Skipping Salah lookup — no merged table.")
else:
    idx = players.index
    salah = players[
        idx.get_level_values("player").str.contains("Salah", case=False)
        & idx.get_level_values("team").str.contains("Liverpool", case=False)
        & idx.get_level_values("league").str.contains("Premier", case=False)
    ]
    print("Salah rows (Liverpool / Premier League):", len(salah))
    if salah.empty:
        print("No match. All Salah-like rows:")
        all_salah = players[idx.get_level_values("player").str.contains("Salah", case=False)]
        display(all_salah)
    else:
        print(salah.T.to_string())


Skipping Salah lookup — no merged table.

In [12]:
# docs/metrics.md "Source (FBref field)" names → FBref MultiIndex matchers.
# Match original (group, leaf) columns from each fetched table, not flattened names.

METRIC_MATCHERS = {
    "goals": lambda g, leaf: leaf == "Gls",
    "pens_made": lambda g, leaf: leaf == "PK",
    "npxg": lambda g, leaf: leaf == "npxG",
    "assists": lambda g, leaf: leaf == "Ast",
    "xg_assist": lambda g, leaf: leaf in {"xAG", "xA", "xg_assist"},
    "shots": lambda g, leaf: leaf == "Sh",
    "progressive_carries": lambda g, leaf: leaf == "PrgC",
    "progressive_passes": lambda g, leaf: leaf == "PrgP",
    "take_ons_won": lambda g, leaf: "take" in g.lower() and leaf in {"Succ", "Successful"},
    "passes_pct": lambda g, leaf: leaf in {"Cmp%", "passes_pct"},
    "touches": lambda g, leaf: leaf == "Touches",
    "tackles_won": lambda g, leaf: leaf == "TklW",
    "interceptions": lambda g, leaf: leaf == "Int",
    "blocks": lambda g, leaf: leaf == "Blocks",
    "aerials_won": lambda g, leaf: "aerial" in g.lower() and leaf == "Won",
}

if players is None or not fetched:
    print("Skipping metrics.md column check — no merged table.")
else:
    original_cols: list[tuple[str, str, str]] = []
    for stat_type, frame in fetched.items():
        cols = frame.columns
        if isinstance(cols, pd.MultiIndex):
            for group, leaf in cols:
                original_cols.append((stat_type, str(group), str(leaf)))
        else:
            for leaf in cols:
                original_cols.append((stat_type, "", str(leaf)))

    print("Original (stat_type, group, leaf) columns:")
    for row in original_cols:
        print(f"  {row}")
    print()

    present: list[str] = []
    missing: list[str] = []
    hits: dict[str, list[tuple[str, str, str]]] = {}
    for field, matcher in METRIC_MATCHERS.items():
        matched = [c for c in original_cols if matcher(c[1], c[2])]
        if matched:
            present.append(field)
            hits[field] = matched
        else:
            missing.append(field)

    print("metrics.md fields PRESENT in standard/shooting/misc:")
    for field in present:
        print(f"  {field}: {hits[field]}")
    print()
    print("metrics.md fields MISSING from these three tables:")
    print(" ", missing or "(none)")
    print()
    print("Flattened merged columns (for later math / docs realignment):")
    for col in players.columns:
        print(f"  {col}")


Skipping metrics.md column check — no merged table.

---

## 5. Pivot: Understat for xG-side stats

FBref's public player tables no longer include Opta xG / progression.
`soccerdata` also wraps [Understat](https://understat.com). Cache stays in
the same `SOCCERDATA_DIR` (`data/data/Understat/`). FBref HTML is left
untouched.

Still no per-90, percentiles, or chart in this section.


## 5a. Introspect soccerdata's Understat scraper

Do not assume the API. `dir(sd.Understat)` plus the docstring/signature of
every `read_*` method that could return player season stats.


In [ ]:
import inspect

print("dir(sd.Understat):")
for name in dir(sd.Understat):
    if not name.startswith("_"):
        print(f"  {name}")

print()
print("read_* methods:")
for name in dir(sd.Understat):
    if not name.startswith("read_"):
        continue
    meth = getattr(sd.Understat, name)
    print()
    print(f"=== {name}{inspect.signature(meth)} ===")
    print(inspect.getdoc(meth))


## 5b. Instantiate Understat

Premier League, season 2025/26. soccerdata parses season strings through
`SeasonCode` (same as FBref). Understat's own URLs use the **start year**
(`https://understat.com/league/EPL/2025`). We pass `seasons=2025` and print
what soccerdata stored.


In [ ]:
UST_LEAGUE = "ENG-Premier League"
# Start year of 2025/26. soccerdata's SeasonCode will normalize this.
UST_SEASON = 2025

ust = sd.Understat(leagues=UST_LEAGUE, seasons=UST_SEASON)

print("leagues:", ust.leagues)
print("seasons (parsed):", list(ust.seasons))
print("ust.data_dir:", ust.data_dir)
print("ust.data_dir inside project data/:", _is_inside(ust.data_dir, DATA_DIR))
print("reader type:", type(ust).__mro__[1].__name__)
ust.data_dir


## 5c. Fetch player season stats

`Understat.read_player_season_stats` — no `stat_type`. One table per
league/season, cached as JSON/HTML under `data/data/Understat/`. This is
HTTP (`BaseRequestsReader`), not Chrome.


In [ ]:
ust_players = ust.read_player_season_stats()

print("shape:", ust_players.shape)
print("index names:", list(ust_players.index.names))
print()
print("columns:")
for i, col in enumerate(ust_players.columns, start=1):
    print(f"  {i:2d}. {col}")
print()
print("dtypes:")
print(ust_players.dtypes.to_string())
print()
print("head:")
display(ust_players.head())


In [ ]:
names = ust_players.index.get_level_values("player")
salah = ust_players[names.str.contains("Salah", case=False)]
print("Salah rows:", len(salah))
if salah.empty:
    print("No player name containing 'Salah'. Nearby:")
    print(sorted({n for n in names if n.lower().startswith("s")})[:40])
else:
    print(salah.T.to_string())


## 5d. Column check vs the fields we need from Understat


In [ ]:
# Names we expect on an Understat player-season table (source JSON keys
# mapped by soccerdata). Confirm presence before any math.
UNDERSTAT_WANTED = [
    "xg",
    "np_xg",
    "xa",
    "shots",
    "key_passes",
    "xg_chain",
    "xg_buildup",
    "goals",
    "np_goals",
    "assists",
    "minutes",
    "position",
]

available = set(ust_players.columns) | set(ust_players.index.names)
print("present:")
for name in UNDERSTAT_WANTED:
    print(f"  {name:12s} {'YES' if name in available else 'NO'}")

print()
print("position value counts:")
print(ust_players["position"].value_counts(dropna=False).to_string())
print()
print("n players:", len(ust_players))
